In [2]:
import pandas as pd
import numpy as np
import itertools
from sklearn.model_selection import LeaveOneOut
from sklearn.linear_model import RidgeCV, LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
import xgboost as xgb

In [3]:
folderPath = r"C:\Users\Aaron\OneDrive\Desktop\Capstone\Data\Features"

precipitation_raw = pd.read_csv(folderPath + "\Weather\Precipitation1985_2025.csv")
sunshine_raw = pd.read_csv(folderPath + "\Weather\SunshineDuration1985_2025.csv")
max_temp_raw = pd.read_csv(folderPath + "\Weather\MaxTemp1985_2025.csv")
min_temp_raw = pd.read_csv(folderPath + "\Weather\MinTemp1985_2025.csv")

precipitation_monthly = precipitation_raw.groupby(["Year", "Month"])["Measurement"].mean().reset_index()
sunshine_monthly = sunshine_raw.groupby(["Year", "Month"])["Measurement"].mean().reset_index()
maxTemp_monthly = max_temp_raw.groupby(["Year", "Month"])["Measurement"].mean().reset_index()
minTemp_monthly = min_temp_raw.groupby(["Year", "Month"])["Measurement"].mean().reset_index()

In [4]:
#Yield detrend = Actual tonnes -  Trend predicted tonnes
def detrend_yield(yield_data):

    yield_copy = yield_data.copy()

    x = yield_copy['Year']
    y = yield_copy['Tonnes']

    slope, intercept = np.polyfit(x,y, deg=1)

    #Predicted yield trend 
    yield_copy['Trend'] = slope * yield_copy['Year'] + intercept

    #Actual yield - predicted yield
    yield_copy['Yield_Detrended'] = yield_copy['Tonnes'] - yield_copy['Trend']

    return yield_copy

In [5]:
precip_m, sun_m, tmax_m, tmin_m = precipitation_monthly, sunshine_monthly, maxTemp_monthly, minTemp_monthly

def monthly_features(monthly_avg, colname):
        pivot = monthly_avg.pivot(index='Year', columns='Month', values='Measurement')
        pivot.columns = [f'{colname}_{m}' for m in pivot.columns]
        return pivot.reset_index()

In [6]:
features = (monthly_features(precip_m, 'Precip').merge(monthly_features(sun_m, 'Sun'), on='Year').merge(monthly_features(tmax_m, 'TempMax'), on='Year').merge(monthly_features(tmin_m, 'TempMin'), on='Year'))

features.head()

,Year,Precip_April,Precip_August,Precip_December,Precip_February,Precip_January,Precip_July,Precip_June,Precip_March,Precip_May,...,TempMin_December,TempMin_February,TempMin_January,TempMin_July,TempMin_June,TempMin_March,TempMin_May,TempMin_November,TempMin_October,TempMin_September
0,1985,54.266667,161.200000,114.566667,69.666667,67.533333,84.133333,84.800000,92.933333,65.066667,...,4.600000,3.300000,-0.233333,11.400000,8.466667,2.633333,6.900000,1.800000,8.333333,11.266667
1,1986,48.466667,166.433333,149.866667,5.533333,115.433333,55.600000,82.800000,103.000000,103.833333,...,3.166667,-0.166667,2.000000,11.733333,9.966667,2.533333,6.933333,4.700000,7.500000,7.666667
2,1987,74.766667,71.066667,93.600000,72.133333,51.200000,38.766667,114.733333,85.166667,22.366667,...,5.633333,2.266667,1.966667,12.066667,9.133333,2.400000,6.333333,4.833333,6.066667,9.266667
3,1988,41.666667,94.133333,59.000000,70.100000,182.633333,111.833333,47.633333,103.800000,84.100000,...,6.400000,2.300000,3.033333,10.666667,10.700000,4.400000,7.066667,4.666667,7.733333,9.700000
4,1989,66.066667,84.266667,124.200000,85.000000,70.633333,12.800000,62.400000,115.966667,15.100000,...,3.633333,3.566667,4.900000,13.233333,9.566667,3.600000,8.066667,5.500000,9.033333,10.166667


In [7]:
springWheat_yield = pd.read_csv(folderPath + r"\CropYield\SpringWheat1985_2025.csv")[['Year', 'Tonnes']]
yield_df = detrend_yield(springWheat_yield)

yield_df.head()

,Year,Tonnes,Trend,Yield_Detrended
0,1985,5.8,6.610453,-0.810453
1,1986,5.4,6.648101,-1.248101
2,1987,6.2,6.685749,-0.485749
3,1988,7.0,6.723397,0.276603
4,1989,6.2,6.761045,-0.561045


In [8]:
data = features.merge(yield_df, on='Year').dropna().sort_values('Year').reset_index(drop=True)

data.head()

,Year,Precip_April,Precip_August,Precip_December,Precip_February,Precip_January,Precip_July,Precip_June,Precip_March,Precip_May,...,TempMin_July,TempMin_June,TempMin_March,TempMin_May,TempMin_November,TempMin_October,TempMin_September,Tonnes,Trend,Yield_Detrended
0,1985,54.266667,161.200000,114.566667,69.666667,67.533333,84.133333,84.800000,92.933333,65.066667,...,11.400000,8.466667,2.633333,6.900000,1.800000,8.333333,11.266667,5.8,6.610453,-0.810453
1,1986,48.466667,166.433333,149.866667,5.533333,115.433333,55.600000,82.800000,103.000000,103.833333,...,11.733333,9.966667,2.533333,6.933333,4.700000,7.500000,7.666667,5.4,6.648101,-1.248101
2,1987,74.766667,71.066667,93.600000,72.133333,51.200000,38.766667,114.733333,85.166667,22.366667,...,12.066667,9.133333,2.400000,6.333333,4.833333,6.066667,9.266667,6.2,6.685749,-0.485749
3,1988,41.666667,94.133333,59.000000,70.100000,182.633333,111.833333,47.633333,103.800000,84.100000,...,10.666667,10.700000,4.400000,7.066667,4.666667,7.733333,9.700000,7.0,6.723397,0.276603
4,1989,66.066667,84.266667,124.200000,85.000000,70.633333,12.800000,62.400000,115.966667,15.100000,...,13.233333,9.566667,3.600000,8.066667,5.500000,9.033333,10.166667,6.2,6.761045,-0.561045


In [9]:
X = data.drop(columns=['Year', 'Tonnes', 'Trend', 'Yield_Detrended'])
y = data['Yield_Detrended']

In [10]:
X.head()

,Precip_April,Precip_August,Precip_December,Precip_February,Precip_January,Precip_July,Precip_June,Precip_March,Precip_May,Precip_November,...,TempMin_December,TempMin_February,TempMin_January,TempMin_July,TempMin_June,TempMin_March,TempMin_May,TempMin_November,TempMin_October,TempMin_September
0,54.266667,161.200000,114.566667,69.666667,67.533333,84.133333,84.800000,92.933333,65.066667,59.833333,...,4.600000,3.300000,-0.233333,11.400000,8.466667,2.633333,6.900000,1.800000,8.333333,11.266667
1,48.466667,166.433333,149.866667,5.533333,115.433333,55.600000,82.800000,103.000000,103.833333,119.466667,...,3.166667,-0.166667,2.000000,11.733333,9.966667,2.533333,6.933333,4.700000,7.500000,7.666667
2,74.766667,71.066667,93.600000,72.133333,51.200000,38.766667,114.733333,85.166667,22.366667,63.933333,...,5.633333,2.266667,1.966667,12.066667,9.133333,2.400000,6.333333,4.833333,6.066667,9.266667
3,41.666667,94.133333,59.000000,70.100000,182.633333,111.833333,47.633333,103.800000,84.100000,42.266667,...,6.400000,2.300000,3.033333,10.666667,10.700000,4.400000,7.066667,4.666667,7.733333,9.700000
4,66.066667,84.266667,124.200000,85.000000,70.633333,12.800000,62.400000,115.966667,15.100000,55.766667,...,3.633333,3.566667,4.900000,13.233333,9.566667,3.600000,8.066667,5.500000,9.033333,10.166667


In [11]:
y.head()

0   -0.810453
1   -1.248101
2   -0.485749
3    0.276603
4   -0.561045
Name: Yield_Detrended, dtype: float64

In [12]:
loo = LeaveOneOut()

def best_hyperparameters(X, y):
  
    hyperparameter_grid = {
        'max_depth': [1, 2, 3, 4, 5],
        'n_estimators': [10,20,30,40],
        'reg_alpha': [0.5, 1.0, 2.0],
        'reg_lambda': [0.5, 1.0, 2.0],
    }
    
    best_result = {'r2': -np.inf}
 
    for max_depth in hyperparameter_grid['max_depth']:
        for n_estimators in hyperparameter_grid['n_estimators']:
            for reg_alpha in hyperparameter_grid['reg_alpha']:
                for reg_lambda in hyperparameter_grid['reg_lambda']:
 
                    params = {
                        'max_depth': max_depth,
                        'n_estimators': n_estimators,
                        'reg_alpha': reg_alpha,
                        'reg_lambda': reg_lambda,
                    }
 
                    y_true = []
                    y_pred = []

                    for train_index, test_index in loo.split(X):

                        model = xgb.XGBRegressor(random_state=42, **params)
                        model.fit(X.iloc[train_index], y.iloc[train_index])
                        y_pred.append(model.predict(X.iloc[test_index])[0])
                        y_true.append(y.iloc[test_index].values[0])
 
                    r2 = r2_score(y_true, y_pred)
 
                    if r2 > best_result['r2']:
                        best_result = {'params': params, 'r2': r2}
 
    print(f"Best hyperparameters: {best_result['params']}")
    print(f"R2: {best_result['r2']:.3f}")

    return best_result['params']

In [13]:
params_monthly = best_hyperparameters(X, y)

Best hyperparameters: {'max_depth': 4, 'n_estimators': 20, 'reg_alpha': 2.0, 'reg_lambda': 2.0}
R2: -0.094


In [14]:
y_true = []
y_pred = []

for train_index, test_index in loo.split(X):
    model = xgb.XGBRegressor(random_state=42, **params_monthly)
    model.fit(X.iloc[train_index], y.iloc[train_index])

    y_pred.append(model.predict(X.iloc[test_index])[0])
    y_true.append(y.iloc[test_index].values[0])

rmse = np.sqrt(mean_squared_error(y_true, y_pred))
r2 = r2_score(y_true, y_pred)

print(f"All Months")
print(f"Features: {X.shape[1]}")
print(f"RMSE: {rmse:.3f}")
print(f"R2: {r2:.3f}")

All Months
Features: 48
RMSE: 0.703
R2: -0.094


In [15]:
critical_months = {
    'Establishment': ['March', 'April'],
    'Flowering': ['July', 'August'],
    }

critical_months = [m for months in critical_months.values() for m in months]

def critical_features(monthly_avg, colname):
    df = monthly_avg[monthly_avg['Month'].isin(critical_months)]

    pivot = df.pivot(index='Year', columns='Month', values='Measurement')
    pivot.columns = [f'{colname}_{m}' for m in pivot.columns]
    return pivot.reset_index()

    
c_features = (critical_features(precip_m, 'Precip')
            .merge(critical_features(sun_m, 'Sun'), on='Year')
            .merge(critical_features(tmax_m, 'TempMax'), on='Year')
            .merge(critical_features(tmin_m, 'TempMin'), on='Year'))

data = c_features.merge(yield_df, on='Year').dropna().sort_values('Year').reset_index(drop=True)

X_critical = data.drop(columns=['Year', 'Tonnes', 'Trend', 'Yield_Detrended'])
y_critical = data['Yield_Detrended']


In [16]:
y_true = []
y_pred = []

for train_index, test_index in loo.split(X_critical):
    model = xgb.XGBRegressor(random_state=42, **params_monthly)
    model.fit(X_critical.iloc[train_index], y_critical.iloc[train_index])

    y_pred.append(model.predict(X_critical.iloc[test_index])[0])
    y_true.append(y_critical.iloc[test_index].values[0])

rmse = np.sqrt(mean_squared_error(y_true, y_pred))
r2 = r2_score(y_true, y_pred)

print(f"Critical Months")
print(f"Features: {X_critical.shape[1]}")
print(f"RMSE: {rmse:.3f}")
print(f"R2: {r2:.3f}")

Critical Months
Features: 16
RMSE: 0.723
R2: -0.157


In [17]:
k=5
y_true = []
y_pred = []


for train_index, test_index in loo.split(X):

    X_train, y_train = X.iloc[train_index], y.iloc[train_index]
    correlations = X_train.corrwith(y_train).abs().sort_values(ascending=False)
    top_features = correlations.head(k).index.tolist()

    model = xgb.XGBRegressor(random_state=42, **params_monthly)
    model.fit(X_train[top_features], y_train)
    y_pred.append(model.predict(X.iloc[test_index][top_features])[0])
    y_true.append(y.iloc[test_index].values[0])

rmse = np.sqrt(mean_squared_error(y_true, y_pred))
r2 = r2_score(y_true, y_pred)

print(f"Top {k} monthly features")
print(f"RMSE: {rmse:.3f}")
print(f"R2: {r2:.3f}")

Top 5 monthly features
RMSE: 0.682
R2: -0.030


In [18]:
season_map = {
        'September': 'Autumn', 'October': 'Autumn', 'November': 'Autumn',
        'December': 'Winter', 'January': 'Winter', 'February': 'Winter',
        'March': 'Spring', 'April': 'Spring', 'May': 'Spring',
        'June': 'Summer', 'July': 'Summer', 'August': 'Summer',
    }

roll_forward = ['September', 'October', 'November', 'December']


def weather_station(weather_data, colname, station):
    df = weather_data.copy()

    if station == 'mean':
        monthly = df.groupby(['Year', 'Month'])['Measurement'].mean().reset_index()

    elif station == 'median':
        monthly = df.groupby(['Year', 'Month'])['Measurement'].median().reset_index()
    
    else:
        monthly = df[df['Location'] == station][['Year', 'Month', 'Measurement']].reset_index(drop=True)

    monthly['Season'] = monthly['Month'].map(season_map)

    rollOn_month = monthly['Month'].isin(roll_forward)
    monthly['HarvestYear'] = np.where(rollOn_month, monthly['Year'] + 1, monthly['Year'])

    season_avg = monthly.groupby(['HarvestYear', 'Season'])['Measurement'].mean().reset_index()
    
    pivot = season_avg.pivot(index='HarvestYear', columns='Season', values='Measurement')
    pivot.columns = [f'{colname}_{s}' for s in pivot.columns]

    return pivot.reset_index().rename(columns={'HarvestYear': 'Year'})

In [19]:
def build_features(station):
    return (weather_station(precipitation_raw, 'Precip', station)
            .merge(weather_station(sunshine_raw, 'Sun', station), on='Year')
            .merge(weather_station(max_temp_raw, 'TempMax', station), on='Year')
            .merge(weather_station(min_temp_raw, 'TempMin', station), on='Year'))

options = ['mean', 'median', 'Shannon Airport', 'Dublin Airport', 'Cork Airport']

print("Weather Station Comparison")

for option in options:
    features = build_features(option)
    data = features.merge(yield_df, on='Year').dropna().sort_values('Year').reset_index(drop=True)
    
    X = data.drop(columns=['Year', 'Tonnes', 'Trend', 'Yield_Detrended'])
    y = data['Yield_Detrended']

    y_true = []
    y_pred = []

    for train_index, test_index in loo.split(X):
        model = xgb.XGBRegressor(random_state=42, **params_monthly)
        model.fit(X.iloc[train_index], y.iloc[train_index])

        y_pred.append(model.predict(X.iloc[test_index])[0])
        y_true.append(y.iloc[test_index].values[0])

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)

    print(f"{option}: RMSE:{rmse:.3f} R2:{r2:.3f}")

Weather Station Comparison
mean: RMSE:0.695 R2:-0.081
median: RMSE:0.713 R2:-0.139
Shannon Airport: RMSE:0.714 R2:-0.144
Dublin Airport: RMSE:0.665 R2:0.007
Cork Airport: RMSE:0.677 R2:-0.027


In [20]:
seasonal_features = build_features('mean')
trended = seasonal_features.merge(springWheat_yield, on='Year').dropna().sort_values('Year').reset_index(drop=True)

X_trended = trended.drop(columns=['Year', 'Tonnes'])
y_trended = trended['Tonnes']  
 
y_true = []
y_pred = []
 
for train_index, test_index in loo.split(X_trended):
    model = xgb.XGBRegressor(random_state=42, **params_monthly)
    model.fit(X_trended.iloc[train_index], y_trended.iloc[train_index])
 
    y_pred.append(model.predict(X_trended.iloc[test_index])[0])
    y_true.append(y_trended.iloc[test_index].values[0])
 
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
r2 = r2_score(y_true, y_pred)
 
print(f"No Detrending")
print(f"RMSE: {rmse:.3f}")
print(f"R2: {r2:.3f}")

No Detrending
RMSE: 0.821
R2: -0.115


In [21]:
data_seasonal = seasonal_features.merge(yield_df, on='Year').dropna().sort_values('Year').reset_index(drop=True)

X_seasonal = data_seasonal.drop(columns=['Year', 'Tonnes', 'Trend', 'Yield_Detrended'])
y_seasonal = data_seasonal['Yield_Detrended']


In [22]:
params_seasonal = best_hyperparameters(X_seasonal, y_seasonal)

Best hyperparameters: {'max_depth': 1, 'n_estimators': 10, 'reg_alpha': 0.5, 'reg_lambda': 1.0}
R2: 0.227


In [23]:
y_true = []
y_pred = []
 
for train_index, test_index in loo.split(X_seasonal):
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_seasonal.iloc[train_index])
    X_test_scaled = scaler.transform(X_seasonal.iloc[test_index])
 
    model = RidgeCV(alphas=np.logspace(-3, 3, 50))
    model.fit(X_train_scaled, y_seasonal.iloc[train_index])
 
    y_pred.append(model.predict(X_test_scaled)[0])
    y_true.append(y_seasonal.iloc[test_index].values[0])
 
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
r2 = r2_score(y_true, y_pred)
 
print("Ridge")
print(f"RMSE: {rmse:.3f}")
print(f"R2: {r2:.3f}")

Ridge
RMSE: 0.756
R2: -0.280


In [24]:
y_true = []
y_pred = []
 
for train_index, test_index in loo.split(X_seasonal):
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_seasonal.iloc[train_index])
    X_test_scaled = scaler.transform(X_seasonal.iloc[test_index])
 
    model = LassoCV(alphas=np.logspace(-3, 1, 50), max_iter=5000)
    model.fit(X_train_scaled, y_seasonal.iloc[train_index])
 
    y_pred.append(model.predict(X_test_scaled)[0])
    y_true.append(y_seasonal.iloc[test_index].values[0])
 
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
r2 = r2_score(y_true, y_pred)
 
print("Lasso")
print(f"RMSE: {rmse:.3f}")
print(f"R2: {r2:.3f}")

Lasso
RMSE: 0.722
R2: -0.169


In [25]:
meanTemp = pd.read_csv(folderPath + "\Weather\AirTemperature1985_2025.csv") 
meanTemp_seasonal = weather_station(meanTemp, 'TempMean', 'mean')
 
features_meanTemp = (weather_station(precipitation_raw, 'Precip', 'mean')
                       .merge(weather_station(sunshine_raw, 'Sun', 'mean'), on='Year')
                       .merge(meanTemp_seasonal, on='Year'))

In [26]:
data = features_meanTemp.merge(yield_df, on='Year').dropna().sort_values('Year').reset_index(drop=True)
X_temp_test = data.drop(columns=['Year', 'Tonnes', 'Trend', 'Yield_Detrended'])
y_temp_test = data['Yield_Detrended']
 
y_true = []
y_pred = []
 
for train_index, test_index in loo.split(X_temp_test):
    model = xgb.XGBRegressor(random_state=42, **params_seasonal)
    model.fit(X_temp_test.iloc[train_index], y_temp_test.iloc[train_index])
 
    y_pred.append(model.predict(X_temp_test.iloc[test_index])[0])
    y_true.append(y_temp_test.iloc[test_index].values[0])
 
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
r2 = r2_score(y_true, y_pred)
 
print(f"Mean Temp")
print(f"Features: {X_temp_test.shape[1]}")
print(f"RMSE: {rmse:.3f}")
print(f"R2: {r2:.3f}")

Mean Temp
Features: 12
RMSE: 0.588
R2: 0.225


In [27]:
data = seasonal_features.merge(yield_df, on='Year').dropna().sort_values('Year').reset_index(drop=True)
X_temp_test = data.drop(columns=['Year', 'Tonnes', 'Trend', 'Yield_Detrended'])
y_temp_test = data['Yield_Detrended']
 
y_true = []
y_pred = []
 
for train_index, test_index in loo.split(X_temp_test):
    model = xgb.XGBRegressor(random_state=42, **params_seasonal)
    model.fit(X_temp_test.iloc[train_index], y_temp_test.iloc[train_index])
 
    y_pred.append(model.predict(X_temp_test.iloc[test_index])[0])
    y_true.append(y_temp_test.iloc[test_index].values[0])
 
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
r2 = r2_score(y_true, y_pred)
 
print(f"Max and Min Temp")
print(f"Features: {X_temp_test.shape[1]}")
print(f"RMSE: {rmse:.3f}")
print(f"R2: {r2:.3f}")

Max and Min Temp
Features: 16
RMSE: 0.587
R2: 0.227


In [28]:
meanTemp = pd.read_csv(folderPath + r"\Weather\AirTemperature1985_2025.csv")
 
def build_features_meanTemp(station):
    return (weather_station(precipitation_raw, 'Precip', station)
            .merge(weather_station(sunshine_raw, 'Sun', station), on='Year')
            .merge(weather_station(meanTemp, 'TempMean', station), on='Year'))
 
def build_features_maxMinTemp(station):
    return (weather_station(precipitation_raw, 'Precip', station)
            .merge(weather_station(sunshine_raw, 'Sun', station), on='Year')
            .merge(weather_station(max_temp_raw, 'TempMax', station), on='Year')
            .merge(weather_station(min_temp_raw, 'TempMin', station), on='Year'))
 
features_meanTemp_all = build_features_meanTemp('mean')
features_maxMinTemp_all = build_features_maxMinTemp('mean')

In [31]:
springWheat = pd.read_csv(folderPath + r"\CropYield\SpringWheat1985_2025.csv")[['Year', 'Tonnes']]
winterWheat = pd.read_csv(folderPath + r"\CropYield\WinterWheat1985_2025.csv")[['Year', 'Tonnes']]
springBarley = pd.read_csv(folderPath + r"\CropYield\SpringBarley1985_2025.csv")[['Year', 'Tonnes']]
winterBarley = pd.read_csv(folderPath + r"\CropYield\WinterBarley1985_2025.csv")[['Year', 'Tonnes']]
springOats = pd.read_csv(folderPath + r"\CropYield\SpringOats1985_2025.csv")[['Year', 'Tonnes']]
winterOats = pd.read_csv(folderPath + r"\CropYield\WinterOats1985_2025.csv")[['Year', 'Tonnes']]
beansAndPeas = pd.read_csv(folderPath + r"\CropYield\BeansandPeas1985_2025.csv")[['Year', 'Tonnes']]
potatoes = pd.read_csv(folderPath + r"\CropYield\Potatoes1985_2025.csv")[['Year', 'Tonnes']]
oilseedRape = pd.read_csv(folderPath + r"\CropYield\OilseedRape1985_2025.csv")[['Year', 'Tonnes']]

crops = [springWheat, winterWheat, springBarley, winterBarley, springOats, winterOats, beansAndPeas, potatoes,oilseedRape]

In [33]:
crops_data = [
    ('SpringWheat', springWheat),
    ('WinterWheat', winterWheat),
    ('SpringBarley', springBarley),
    ('WinterBarley', winterBarley),
    ('SpringOats', springOats),
    ('WinterOats', winterOats),
    ('BeansAndPeas', beansAndPeas),
    ('Potatoes', potatoes),
    ('OilseedRape', oilseedRape),
]

In [ ]:

comparison_results = []
 
for crop_name, crop in crops_data:
    crop_yield_detrended = detrend_yield(crop)
 
    print(f"\n{crop_name}")
 
    for label, feature_table in [('Mean Temp', features_meanTemp_all), ('Max+Min Temp', features_maxMinTemp_all)]:
 
        data = feature_table.merge(crop_yield_detrended, on='Year').dropna().sort_values('Year').reset_index(drop=True)
        X_temp_test = data.drop(columns=['Year', 'Tonnes', 'Trend', 'Yield_Detrended'])
        y_temp_test = data['Yield_Detrended']
 
        y_true = []
        y_pred = []
 
        for train_index, test_index in loo.split(X_temp_test):
            model = xgb.XGBRegressor(random_state=42, **params_seasonal)
            model.fit(X_temp_test.iloc[train_index], y_temp_test.iloc[train_index])
 
            y_pred.append(model.predict(X_temp_test.iloc[test_index])[0])
            y_true.append(y_temp_test.iloc[test_index].values[0])
 
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
        r2 = r2_score(y_true, y_pred)
 
        print(f"{label:15s} RMSE={rmse:.3f}, R2={r2:.3f}")
 
        comparison_results.append({'crop': crop_name, 'temperature_type': label, 'r2': r2, 'rmse': rmse})
 
comparison_table = pd.DataFrame(comparison_results)
print("\n\nR2 comparison: Mean Temp vs Max+Min Temp")
print(comparison_table.pivot(index='crop', columns='temperature_type', values='r2').round(3))


SpringWheat
  Mean Temp       RMSE=0.588, R2=0.225
  Max+Min Temp    RMSE=0.587, R2=0.227

WinterWheat
  Mean Temp       RMSE=0.886, R2=-0.012
  Max+Min Temp    RMSE=0.889, R2=-0.019

SpringBarley
  Mean Temp       RMSE=0.504, R2=0.250
  Max+Min Temp    RMSE=0.501, R2=0.259

WinterBarley
  Mean Temp       RMSE=0.693, R2=-0.073
  Max+Min Temp    RMSE=0.674, R2=-0.014

SpringOats
  Mean Temp       RMSE=0.438, R2=0.211
  Max+Min Temp    RMSE=0.446, R2=0.185

WinterOats
  Mean Temp       RMSE=0.463, R2=-0.017
  Max+Min Temp    RMSE=0.474, R2=-0.067

BeansAndPeas
  Mean Temp       RMSE=1.302, R2=-0.010
  Max+Min Temp    RMSE=1.419, R2=-0.200

Potatoes
  Mean Temp       RMSE=2.852, R2=0.425
  Max+Min Temp    RMSE=3.018, R2=0.356

OilseedRape
  Mean Temp       RMSE=0.825, R2=-0.113
  Max+Min Temp    RMSE=0.860, R2=-0.210


R2 comparison: Mean Temp vs Max+Min Temp
temperature_type  Max+Min Temp  Mean Temp
crop                                     
BeansAndPeas            -0.200     -0.010
Oils

In [36]:
growth_year = ['September', 'October', 'November', 'December','January', 'February', 'March', 'April','May', 'June', 'July', 'August']
window_width = 4   
window_results = []
 

roll_forward_months = ['September', 'October', 'November', 'December']

for start_position in range(len(growth_year)):
    window = [growth_year[(start_position + offset) % 12] for offset in range(window_width)]
 
    def build_window_feature(monthly_avg, colname):
        df = monthly_avg[monthly_avg['Month'].isin(window)].copy()
 
        rollOn_month = df['Month'].isin(roll_forward_months)
        df['HarvestYear'] = np.where(rollOn_month, df['Year'] + 1, df['Year'])
 
        window_avg = df.groupby('HarvestYear')['Measurement'].mean().reset_index()
        window_avg = window_avg.rename(columns={'Measurement': colname, 'HarvestYear': 'Year'})
        return window_avg
 
    window_features = (build_window_feature(precip_m, 'Precip')
                        .merge(build_window_feature(sun_m, 'Sun'), on='Year')
                        .merge(build_window_feature(tmax_m, 'TempMax'), on='Year')
                        .merge(build_window_feature(tmin_m, 'TempMin'), on='Year'))
 
    data = window_features.merge(yield_df, on='Year').dropna().sort_values('Year').reset_index(drop=True)
    X_window = data.drop(columns=['Year', 'Tonnes', 'Trend', 'Yield_Detrended'])
    y_window = data['Yield_Detrended']
 
    y_true = []
    y_pred = []

    for train_index, test_index in loo.split(X_window):
        
        model = xgb.XGBRegressor(random_state=42, **params_seasonal)
        model.fit(X_window.iloc[train_index], y_window.iloc[train_index])

        y_pred.append(model.predict(X_window.iloc[test_index])[0])
        y_true.append(y_window.iloc[test_index].values[0])
 
    r2 = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
 
    window_months = f"{window[0]}-{window[-1]}"
    window_results.append({'window': window_months, 'months': window, 'rmse': rmse, 'r2': r2})
 
results_df = pd.DataFrame(window_results).sort_values('r2', ascending=False)

print("Ranked by R2")
print(results_df[['window', 'r2', 'rmse']])

Ranked by R2
                window        r2      rmse
8           May-August  0.297919  0.563138
7           April-July  0.144280  0.621709
9       June-September  0.133867  0.625480
10        July-October  0.114085  0.632583
11     August-November  0.062470  0.650749
6           March-June  0.026131  0.663241
1      October-January -0.019907  0.678737
3       December-March -0.052048  0.689349
0   September-December -0.055646  0.686280
2    November-February -0.136505  0.716485
5         February-May -0.158900  0.723509
4        January-April -0.167408  0.726160


In [39]:
window = ['May', 'June', 'July', 'August']

def build_window(monthly_avg, colname):
    df = monthly_avg[monthly_avg['Month'].isin(window)].copy()
    window_avg = df.groupby('Year')['Measurement'].mean().reset_index()
    window_avg = window_avg.rename(columns={'Measurement': colname})
    return window_avg

window_features = (build_window(precip_m, 'Precip')
                    .merge(build_window(sun_m, 'Sun'), on='Year')
                    .merge(build_window(tmax_m, 'TempMax'), on='Year')
                    .merge(build_window(tmin_m, 'TempMin'), on='Year'))

data = window_features.merge(yield_df, on='Year').dropna().sort_values('Year').reset_index(drop=True)
X_window = data.drop(columns=['Year', 'Tonnes', 'Trend', 'Yield_Detrended'])
y_window = data['Yield_Detrended']

In [40]:
y_true = []
y_pred = []

for train_index, test_index in loo.split(X_window):
    model = xgb.XGBRegressor(random_state=42, **params_seasonal)
    model.fit(X_window.iloc[train_index], y_window.iloc[train_index])
    y_pred.append(model.predict(X_window.iloc[test_index])[0])
    y_true.append(y_window.iloc[test_index].values[0])

observed_r2 = r2_score(y_true, y_pred)

In [ ]:
shuffles = 2000
random_number = np.random.default_rng(42)
r2_scores = []

for shuffle_number in range(shuffles):
    shuffled_y = pd.Series(random_number.permutation(y_window.values), index=y_window.index)

    y_true = []
    y_pred = []

    for train_index, test_index in loo.split(X_window):
        model = xgb.XGBRegressor(random_state=42, **params_seasonal)
        model.fit(X_window.iloc[train_index], shuffled_y.iloc[train_index])
        y_pred.append(model.predict(X_window.iloc[test_index])[0])
        y_true.append(shuffled_y.iloc[test_index].values[0])

    r2_scores.append(r2_score(y_true, y_pred))

r2_scores = np.array(r2_scores)
p_value = (np.sum(r2_scores >= observed_r2) + 1) / (shuffles + 1)

print(f"Observed R2 (May-August): {observed_r2:.3f}")
print(f"Permuted R2 mean: {r2_scores.mean():.3f}, max: {r2_scores.max():.3f}")
print(f"p-value: {p_value:.4f}")

Observed R2 (May-August): 0.298
Permuted R2 mean: -0.127, max: 0.387
p-value: 0.0010
Significant at p<0.05: True
